# MediSense AI — End-to-*End* (Medical Intelligence AI Platform)


----



In [5]:
!pip install numpy==1.26.4 tensorflow==2.19.0

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached tensorflow-2.19.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.11.2-py3-none-any.whl.metadata (5.9 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached tensorflow-2.19.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (644.9 MB)
Using cached keras-3.11.2-py3-none-any.whl (1.4 MB)
Using cached tensorboard-2.19.0-py3-none-any.whl (5.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.12.3
    Uninstalling tensorboard-2.12.3:
      Successfully uninstalled tensorboard-2.12.3
  Attempting uninstall

In [6]:
!pip install --quiet kaggle==1.5.12 scikit-learn==1.3.2 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.12.2 plotly==5.15.0 tensorflow==2.12.0 torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install --quiet albumentations==1.3.0 opencv-python==4.8.0.76 pillow==9.5.0 tqdm==4.65.0 nbformat==5.7.0
!pip install --quiet transformers==4.30.2 datasets==2.13.0 accelerate==0.20.3 sentencepiece==0.1.99 spacy[lookups]==3.5.3 spacy-transformers==1.2.4
!pip install --quiet grad-cam==1.4.5 streamlit==1.24.0 fastapi==0.100.0 uvicorn[standard]==0.22.0 xgboost==1.7.6
!pip install --quiet protobuf==4.25.3 numpy==1.24.3 scipy==1.10.1 jax==0.4.13 flax==0.6.11 ydf==0.10.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy-transformers 1.3.9 requires transformers<4.50.0,>=3.4.0, but you have transformers 4.55.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
treescope 0.1.10 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
geopandas 1.1.1 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
geopandas 1.1.1 requires pandas>=2.0.0, but you have pandas 1.5.3 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.12.0 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.12.0 which is incompatible.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, but you have scikit-learn 1.3.2 which is incompatible.
mizani 0.13.5 requ

## 2) Kaggle authentication and dataset download


In [6]:
from pathlib import Path
kaggle_json_path = '/content/kaggle.json'
home_kaggle = Path.home() / '.kaggle'
home_kaggle.mkdir(exist_ok=True)

if Path(kaggle_json_path).exists():
    import shutil
    shutil.copy(kaggle_json_path, home_kaggle / 'kaggle.json')
    os.chmod(home_kaggle / 'kaggle.json', 0o600)
    print('kaggle.json copied to ~/.kaggle/')
else:
    print('No kaggle.json found at /content/kaggle.json. Use the file upload in Colab to place it there.')

No kaggle.json found at /content/kaggle.json. Use the file upload in Colab to place it there.


In [7]:
BASE_DIR = Path('/content/medisense_data')
for sub in ['tabular', 'images', 'nlp', 'models', 'notebooks', 'exports']:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)
print('Data directories:', list(BASE_DIR.glob('*')))

Data directories: [PosixPath('/content/medisense_data/tabular'), PosixPath('/content/medisense_data/models'), PosixPath('/content/medisense_data/exports'), PosixPath('/content/medisense_data/notebooks'), PosixPath('/content/medisense_data/nlp'), PosixPath('/content/medisense_data/images')]


### Tabular ML example — Pima Diabetes (quick demo)

In [8]:
import os
pima_paths = [
    '/content/data/diabetes.csv', '/content/data/diabetes.csv.zip',
    str(BASE_DIR / 'tabular' / 'diabetes.csv')
]
pima_path = next((p for p in pima_paths if os.path.exists(p)), None)
if pima_path is None:
    print('Pima diabetes dataset not found locally. Please download from Kaggle and place diabetes.csv in /content/data or medisense_data/tabular.')
else:
    df = pd.read_csv(pima_path)
    display(df.head())
    print('\nBasic shape:', df.shape)

    print('\nNull values:\n', df.isnull().sum())
    sns.pairplot(df.sample(200), hue='Outcome', vars=df.columns[:-1])
    plt.show()


    from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report

    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    print('Test accuracy:', accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

Pima diabetes dataset not found locally. Please download from Kaggle and place diabetes.csv in /content/data or medisense_data/tabular.


### Medical Image DL — Chest X-Ray (Transfer Learning + Grad-CAM)

In [ ]:
!pip install --quiet numpy==1.23.5
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [1]:
import os

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, ResNet50
from tensorflow.keras import layers, models, optimizers, callbacks

IMG_SIZE = (224,224)
BATCH = 16
train_dir = '/content/data/chest_xray/train'
val_dir = '/content/data/chest_xray/val'
test_dir = '/content/data/chest_xray/test'

if not os.path.exists(train_dir):
    print('Chest X-Ray dataset not found. Download and set train/val/test paths.')
else:
    train_gen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=15,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   shear_range=0.1,
                                   zoom_range=0.1,
                                   horizontal_flip=True,
                                   fill_mode='nearest')

    val_gen = ImageDataGenerator(rescale=1./255)

    train_flow = train_gen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH, class_mode='binary')
    val_flow = val_gen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH, class_mode='binary')

    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE,3))
    base.trainable = False
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=base.input, outputs=out)
    model.compile(optimizer=optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    es = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(train_flow, validation_data=val_flow, epochs=20, callbacks=[es])
    model.save('/content/medisense_data/models/chest_xray_effnet.h5')

    from tf_explain.core.grad_cam import GradCAM
    explainer = GradCAM()

    import glob
    sample = glob.glob(test_dir + '/*/*.jpeg')[0]
    data = ([plt.imread(sample)], None)
    grid = explainer.explain(data, model, class_index=0)
    plt.imshow(grid); plt.axis('off')

Chest X-Ray dataset not found. Download and set train/val/test paths.


### NLP section — NER & Chatbot skeleton


In [4]:
!pip install torch --upgrade
!pip install transformers --upgrade
!pip install torch==2.3.1 torchvision torchaudio
!pip install torch==2.3.1 torchvision torchaudio


  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse_cu12-12.5.8.93-py3-n

In [2]:
import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load('en_core_web_sm')

text = 'Patient complains of chest pain and shortness of breath. Prescribed aspirin 81 mg daily.'
doc = nlp(text)
print('Entities detected:')
for ent in doc.ents:
    print(ent.text, ent.label_)

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased')
clf = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)
print(clf('Patient has high fever and severe cough.'))

Entities detected:
81 CARDINAL
daily DATE


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.5226895213127136}]


### Model evaluation & utilities


In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

def cross_val_report(model, X, y, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    print('ROC-AUC scores:', scores, 'mean:', scores.mean())

def save_model_sklearn(model, path):
    import joblib
    joblib.dump(model, path)
    print('Saved', path)

### Deployment & cloud notes

In [4]:
fastapi_code = '''
from fastapi import FastAPI, File, UploadFile
from pydantic import BaseModel
import uvicorn
import numpy as np
from PIL import Image
import io

app = FastAPI()

@app.get('/ping')
def ping():
    return {'status':'ok'}

@app.post('/predict-image/')
async def predict_image(file: UploadFile = File(...)):
    contents = await file.read()
    img = Image.open(io.BytesIO(contents)).resize((224,224)).convert('RGB')
    # preprocess and run model prediction here
    return {'prediction': 'placeholder'}

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)
'''
with open('app_fastapi_example.py','w') as f:
    f.write(fastapi_code)
print('FastAPI skeleton saved to app_fastapi_example.py')

FastAPI skeleton saved to app_fastapi_example.py
